# News Agent — Agente de Noticias Financieras

Agente que recolecta, filtra y resume noticias financieras desde feeds RSS de medios internacionales y regionales.

| Versión | Descripción | Estado |
|---------|-------------|--------|
| 0 | Lector de feeds RSS: título, enlace y fecha | COMPLETADA |
| 1 | Filtrado por palabras clave y exportación a CSV | COMPLETADA |
| 2 | Almacenamiento en base de datos y deduplicación | COMPLETADA |
| 3 | Análisis de sentimiento con NLP | COMPLETADA |
| 4 | Agente con LLM: resumen y alertas automáticas | Pendiente |

In [22]:
# Versión 0 — News Agent: lector de feeds RSS de medios financieros.
# Crea la estructura de carpetas del proyecto si aún no existe.
# - data/raw/       → feeds RSS descargados sin procesar
# - data/processed/ → noticias limpias y filtradas
# - logs/           → registros de ejecución
# - config/         → fuentes RSS y parámetros de configuración

from pathlib import Path
import os

base = Path(os.getcwd())
print(f"Creando carpetas en: {base}")

for carpeta in ["data/raw", "data/processed", "logs", "config"]:
    ruta = base / carpeta
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"✓ {ruta}")

#Aquí se crearon las carpetas donde el agente se pondrá a almacenar la información de noticias que vaya recolectando

Creando carpetas en: g:\Data Lake\Noticias
✓ g:\Data Lake\Noticias\data\raw
✓ g:\Data Lake\Noticias\data\processed
✓ g:\Data Lake\Noticias\logs
✓ g:\Data Lake\Noticias\config


In [23]:
#Ahora instalaremos Feedparser
import feedparser
print(f"feedparser {feedparser.__version__} importado correctamente.")

feedparser 6.0.12 importado correctamente.


In [24]:
# Fuentes RSS del agente — medios financieros por cobertura geográfica

FEEDS = [
    {"nombre": "Reuters Business",  "url": "https://news.google.com/rss/search?q=when:24h+allinurl:reuters.com&ceid=US:en&hl=en-US&gl=US"},    # Internacional
    {"nombre": "Yahoo Finance",     "url": "https://finance.yahoo.com/news/rssindex"},            # Internacional (EE.UU.)
    {"nombre": "Expansión México",  "url": "https://expansion.mx/rss"},                           # México
    {"nombre": "Investing España",        "url": "https://es.investing.com/rss/news.rss"}, # España
]

In [25]:
# Recorre cada fuente RSS y muestra las primeras 5 noticias

for feed in FEEDS:

    # Descarga y parsea el feed
    resultado = feedparser.parse(feed["url"])

    # Encabezado del medio
    print(f"\n{'='*60}")
    print(f"  {feed['nombre']}")
    print(f"{'='*60}")

    # Itera sobre las primeras 5 entradas
    for entrada in resultado.entries[:5]:
        titulo = entrada.get("title", "Sin título")
        enlace = entrada.get("link", "Sin enlace")
        fecha  = entrada.get("published", "Fecha no disponible")

        print(f"\nTítulo : {titulo}")
        print(f"Enlace : {enlace}")
        print(f"Fecha  : {fecha}")
        print("-" * 60)


  Reuters Business

Título : Dell shares soar 30% as AI server demand, price hikes power stellar quarter - Reuters
Enlace : https://news.google.com/rss/articles/CBMivAFBVV95cUxPcVJqdDdzRnFib0ZkeU1rd1F6WlZrOHF4cTB2MGVoLUhqRlNJNzY1RXY0MHBpcHhNUXBQODVMSlNRd0c5N292QnRTcUZpRkpvR2dVYnVLSm1YalVEUkZ0blV6UllTZTJPaUtlU2VfbDZjWk1ESWYwcGtwRzAzWHV3UjN0dS1Oc1ExNXNLSHVmRmRBQy1xOWFyN3ZaNjFFdDVzYWxEcGN4UTBmdUs1bV9ySnpBaTVRSkdQMkZqcg?oc=5
Fecha  : Fri, 29 May 2026 15:57:39 GMT
------------------------------------------------------------

Título : Wall Street extends rally on tech strength; Middle East deal stays in focus - Reuters
Enlace : https://news.google.com/rss/articles/CBMitgFBVV95cUxPWmhiS2szMU9CTTVRT2ZFVXdPSlJKa28wbFdjSTNnNm1XbjBpT3paRHJmV2FZWmI1eVlQU3NwNy1BNXJ5ZXc1TzV3NWJJaVlNNUNUR1hLVDllZlVnVXRzVlhBa1d2LTdfT0RXaHRNS2dTandQdUlpWFk2aTJ4RVpyTVlyYWZTUW1MWEdQclg1VElmcUNxRHJFYVVlNDdvbmNvU01palZpOGRQQ0pMbDhUbUVhUVBzdw?oc=5
Fecha  : Fri, 29 May 2026 18:37:27 GMT
-------------------------------------

In [26]:
import subprocess, sys

# Instala python-dotenv si no está disponible en el entorno
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

import os
from dotenv import load_dotenv

# Carga las variables definidas en el archivo .env al entorno del proceso
load_dotenv()

# Lee la API key de Anthropic desde las variables de entorno
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# Confirma que la clave se cargó, mostrando solo los primeros 8 caracteres
if ANTHROPIC_API_KEY:
    print(f"ANTHROPIC_API_KEY cargada correctamente: {ANTHROPIC_API_KEY[:8]}...")
else:
    print("⚠️  ANTHROPIC_API_KEY no encontrada. Verifica que exista en el archivo .env")

ANTHROPIC_API_KEY cargada correctamente: sk-ant-a...


In [27]:
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# --- Recolección de titulares ---
# Recorre todos los feeds y acumula hasta 20 títulos en total
titulos = []
for feed in FEEDS:
    resultado = feedparser.parse(feed["url"])
    contador = 0
    for entrada in resultado.entries:
        if contador >= 5:
            break
        titulo = entrada.get("title", "").strip()
        if titulo:
            titulos.append(f"[{feed['nombre']}] {titulo} | URL: {entrada.get('link', 'No disponible')}")
            contador += 1

# Esta noticias_texto será la lista previamente creada de encabezados de noticias
# Para esta ocasión, lo limitamos a 5 noticias de las 4 fuentes, dándo un total de 20 noticas
noticias_texto = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titulos))

#Todo para que el prompt le pida al agente que de esas 20 noticias
# se seleccionen las 8 más relevantes para los temas que menciona el prompt
#Si quisieramos conocer la lista completa de esas 20 solo habría que ----> print(noticias_texto)
#También le pedimos que incluya el url de la notica por si el usuario desea acceder y leerela completa
prompt = f"""You are a senior financial analyst. I am providing you with a list of headlines from various sources (Reuters Business, Yahoo Finance, Expansión México, and Investing España) riight bellow: You must analyze the entire list provided, regardless of the source, and do not filter by source.

{noticias_texto}

Your task:
1. Select the 8 most relevant news for a finance professional interested in global markets, political situations, the economy, and stock markets.
2. For each selected news item, write a summary of exactly 2 sentences: the first explains the fact, the second explains its financial implication.
3. If the headline is in English, write the summary in English. If it is in Spanish, write it in Spanish.
4. Include the exact URL of the news item you are summarizing.
5. Respond ONLY with the 8 numbered news items, without any introduction or additional text.

Response format:
1. [Headline]
   Summary: [Sentence 1. Sentence 2.]
   URL: [Link]
"""

# --- Llamada a Anthropic y presentación del resultado ---
print("Analizando titulares con Claude...\n")
print("=" * 60)
respuesta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    #estamos usando de momento el modelo más económico en tokens para las pruebas
    max_tokens=2048,
    #De momento ponemos un máximo en el  número de tokens porque solo se está enfocando en hacer un resumen de las noticias como si fuera encabezado
    messages=[{"role": "user", "content": prompt}]
)
print(respuesta.content[0].text)
print("=" * 60)

Analizando titulares con Claude...

1. [Reuters Business] Dell shares soar 30% as AI server demand, price hikes power stellar quarter
   Summary: Dell Technologies experienced a 30% share price increase driven by strong demand for AI servers and successful price increases during the latest quarter. This reflects robust enterprise spending on AI infrastructure and demonstrates Dell's advantageous market positioning to capitalize on the ongoing artificial intelligence investment cycle.
   URL: https://news.google.com/rss/articles/CBMivAFBVV95cUxPcVJqdDdzRnFib0ZkeU1rd1F6WlZrOHF4cTB2MGVoLUhqRlNJNzY1RXY0MHBpcHhNUXBQODVMSlNRd0c5N292QnRTcUZpRkpvR2dVYnVLSm1YalVEUkZ0blV6UllTZTJPaUtlU2VfbDZjWk1ESWYwcGtwRzAzWXV3UjN0dS1Oc1ExNXNLSHVmRmRBQy1xOWFyN3ZaNjFFdDVzYWxEcGN4UTBmdUs1bV9ySnpBaTVRSkdQMkZqcg?oc=5

2. [Reuters Business] Wall Street extends rally on tech strength; Middle East deal stays in focus
   Summary: U.S. stock markets continued their upward trajectory with technology stocks leading the gai

In [28]:
import smtplib
import re
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import date

#Ahora, con esta celda se mandará el output previo a mi correo
#Todo vía la contraseña de "gmail APP"

# --- Credenciales desde el entorno (.env) ---
GMAIL_USER         = os.getenv("GMAIL_USER")
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD")

# --- Asunto con la fecha de hoy ---
asunto = f"Daily Financial News - {date.today().strftime('%Y-%m-%d')}"

# --- Conversión del texto plano a HTML ---
# Toma el texto de la respuesta de Claude y aplica formato HTML básico
texto_plano = respuesta.content[0].text

def texto_a_html(texto):
    # Escapa caracteres reservados de HTML
    texto = texto.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    # Convierte las URLs (líneas "URL: ...") en enlaces clicables
    texto = re.sub(
        r'URL:\s*(https?://\S+)',
        r'URL: <a href="\1" target="_blank">\1</a>',
        texto
    )

    # Pone en negrita las líneas de titular numeradas: "1. [Titular]"
    texto = re.sub(
        r'^(\d+\.\s.+)$',
        r'<b>\1</b>',
        texto,
        flags=re.MULTILINE
    )

    # Reemplaza saltos de línea por <br> para respetar la estructura en HTML
    return texto.replace("\n", "<br>\n")

cuerpo_html = f"""
<html>
  <body style="font-family: Arial, sans-serif; font-size: 14px; line-height: 1.6; color: #222;">
    <h2 style="color: #1a1a2e;">Daily Financial News — {date.today().strftime('%B %d, %Y')}</h2>
    <hr>
    <p>{texto_a_html(texto_plano)}</p>
    <hr>
    <p style="font-size: 11px; color: #888;">Generado automáticamente por News Agent</p>
  </body>
</html>
"""

# --- Construcción del mensaje MIME ---
msg = MIMEMultipart("alternative")
msg["Subject"] = asunto
msg["From"]    = GMAIL_USER
msg["To"]      = GMAIL_USER                    # self-send

# Adjunta versión HTML (los clientes de email la prefieren sobre texto plano)
msg.attach(MIMEText(cuerpo_html, "html", "utf-8"))

# --- Envío por SMTP con SSL (puerto 465) ---
# Requiere una App Password de Google, no la contraseña normal de la cuenta
with smtplib.SMTP_SSL("smtp.gmail.com", 465) as servidor:
    servidor.login(GMAIL_USER, GMAIL_APP_PASSWORD)
    servidor.sendmail(GMAIL_USER, GMAIL_USER, msg.as_string())

print(f"Email enviado correctamente a {GMAIL_USER}")

Email enviado correctamente a bernotsantiago@gmail.com
